# GigaChat v2 на примерах — и что меняется относительно v1

Этот notebook можно читать как небольшую поваренную книгу: **задача → короткий код → ответ модели**.
Каждый пример запускается отдельно после настройки. Ответы сохраняются прямо под ячейками.
Здесь v1 и v2 — маршруты API одной версии `langchain-gigachat`, а не версии LangChain.

Начните с обычного вопроса и сравнения ответов, затем попробуйте streaming, JSON, инструменты, reasoning и файлы.
Вызовы расходуют токены. Ниже есть загрузка готового JPG и создание нового диалога с продолжением по ID.

**Как читать вывод:** под кодом находится настоящий ответ текущего запуска. Ошибка API означает, что пример
не отработал на выбранном endpoint/model; она не заменяется придуманным ответом.

> Блоки «Пример ответа» — иллюстрации формы вывода. Настоящие ответы сохраняются под выполненными ячейками. Новые примеры генерации, загрузки и продолжения диалога нужно запустить отдельно.

## Подготовка

Для предрелиза `langchain-gigachat 0.5.2a1` используйте SDK `0.2.4a1`.
Ниже его исходники находятся в `../gigachat`; замените путь при необходимости.
Из корня репозитория:

```bash
python -m venv .venv-notebook
source .venv-notebook/bin/activate
python -m pip install -e ../gigachat -e ./libs/gigachat jupyterlab ipykernel python-dotenv
python -m ipykernel install --user --name gigachat-notebook --display-name "GigaChat notebook"
python -m jupyterlab examples/api_v2_feature_comparison.ipynb
```

Выберите kernel **GigaChat notebook**. В `.env` задайте `GIGACHAT_CREDENTIALS` или `GIGACHAT_ACCESS_TOKEN`
(также поддерживается пара `GIGACHAT_USER` / `GIGACHAT_PASSWORD`). Секреты не выводятся.
При необходимости задайте `GIGACHAT_BASE_URL` и `GIGACHAT_CA_BUNDLE_FILE`.
Base URL с `/v1` намеренный: SDK самостоятельно выбирает маршрут v2.

После этой настройки достаточно запускать интересующие ячейки; отдельного переключателя RUN_LIVE нет.

In [ ]:
import os
import json
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv
from IPython.display import display, Markdown
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_gigachat import GigaChat

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "libs/gigachat/pyproject.toml").exists())
load_dotenv(ROOT / ".env", override=False)
MODEL = os.getenv("GIGACHAT_V2_EXAMPLE_MODEL", os.getenv("GIGACHAT_MODEL", "GigaChat-3-Ultra"))
BASE_URL = os.getenv("GIGACHAT_BASE_URL", "https://api.giga.chat/v1")
AUTH = {key: os.getenv("GIGACHAT_" + key.upper()) for key in (
    "credentials", "access_token", "user", "password", "scope", "auth_url",
    "ca_bundle_file", "cert_file", "key_file", "key_file_password")}
AUTH = {key: value for key, value in AUTH.items() if value}

def model(v2=True, **options):
    settings = dict(model=MODEL, base_url=BASE_URL, use_api_v2=v2,
                    temperature=0, max_tokens=512, timeout=45, max_retries=0, **AUTH)
    settings.update(options)
    return GigaChat(**settings)

v1 = model(v2=False)
v2 = model(v2=True)
print(f"Модель: {MODEL}")
print(f"langchain-gigachat {version('langchain-gigachat')}; SDK {version('gigachat')}")

### Две небольшие функции для читаемого вывода

`answer()` показывает текст сообщения, `details()` — полезные метаданные без полного дампа объекта.
Все вызовы модели ниже остаются обычным Python-кодом, без тестового runner.

In [ ]:
def answer(message, title="Ответ"):
    display(Markdown(f"**{title}**"))
    display(Markdown(message.text or "_Текстового ответа нет; посмотрите tool_calls/content_blocks._"))

def details(message):
    print("Завершение:", message.response_metadata.get("finish_reason"))
    print("Токены:", message.usage_metadata)

def collect(chunks):
    full = None
    for chunk in chunks:
        print(chunk.text, end="", flush=True)
        full = chunk if full is None else full + chunk
    print()
    if full is None:
        raise RuntimeError("Сервер вернул пустой stream")
    return full

## 1. Один вопрос — два ответа

Попросим объяснить идемпотентность на бытовом примере. Запрос, модель и sampling одинаковые.
Сначала запустите v1, затем v2 и прочитайте оба ответа. Совпадение текста не требуется.

In [ ]:
question = "Объясни идемпотентность на примере кнопки лифта. Не больше трёх предложений."
response_v1 = v1.invoke(question)
answer(response_v1, "API v1")
details(response_v1)

In [ ]:
response_v2 = v2.invoke("Объясни идемпотентность на примере кнопки лифта. Не больше трёх предложений.")
answer(response_v2, "API v2")
details(response_v2)

**Пример ответа — иллюстрация**

Нажать кнопку вызова лифта несколько раз — всё равно что нажать её один раз: лифт уже вызван. Повторное нажатие не создаёт дополнительные поездки.

**Что изменилось в коде:** только `use_api_v2=True`. Тип результата по-прежнему `AIMessage`.
Можно переключить маршрут и для отдельного вызова через `bind`.

In [ ]:
one_off = v1.bind(use_api_v2=True).invoke("Закончи фразу: идемпотентная операция при повторении…")
answer(one_off)

## 2. Ответ появляется по мере генерации

`stream()` отдаёт чанки. Печатаем текст сразу и одновременно складываем чанки, чтобы сохранить итоговое сообщение.
Ниже два самостоятельных примера с одним запросом.

In [ ]:
stream_v1 = collect(v1.stream("Дай три коротких совета по написанию понятных функций Python."))
details(stream_v1)

In [ ]:
stream_v2 = collect(v2.stream("Дай три коротких совета по написанию понятных функций Python."))
details(stream_v2)
print("Модель в итоговых metadata:", stream_v2.response_metadata.get("model"))

**Пример текста, появляющегося постепенно — иллюстрация**

1. Давайте функциям имена, которые объясняют действие.
2. Оставляйте каждой функции одну понятную задачу.
3. Показывайте назначение функции коротким примером.

## 3. Асинхронный запрос

В Jupyter можно писать `await` прямо в ячейке. `asyncio.run()` здесь не нужен.

In [ ]:
async_v1 = await v1.ainvoke("Придумай короткое название для приложения учёта домашних растений.")
answer(async_v1, "API v1 · ainvoke")

In [ ]:
async_v2 = await v2.ainvoke("Придумай короткое название для приложения учёта домашних растений.")
answer(async_v2, "API v2 · ainvoke")

**Пример ответа — иллюстрация**

«Листочек»

### Асинхронный streaming

Тот же подход, но чанки получаем через `async for`.

In [ ]:
async_stream_v1 = None
async for chunk in v1.astream("Напиши двухстрочное поздравление с завершением проекта."):
    print(chunk.text, end="", flush=True)
    async_stream_v1 = chunk if async_stream_v1 is None else async_stream_v1 + chunk
print()
details(async_stream_v1)

In [ ]:
async_stream_v2 = None
async for chunk in v2.astream("Напиши двухстрочное поздравление с завершением проекта."):
    print(chunk.text, end="", flush=True)
    async_stream_v2 = chunk if async_stream_v2 is None else async_stream_v2 + chunk
print()
details(async_stream_v2)

## 4. Получаем объект вместо свободного текста

Извлечём товар и количество из короткой фразы. Pydantic задаёт структуру результата.
`json_schema` — нативный формат API; он существовал и в v1, а в v2 получил новый mapping.
Поддержка зависит от модели.

In [ ]:
class Product(BaseModel):
    """Товар из сообщения покупателя."""
    name: str = Field(description="Название товара в единственном числе")
    count: int = Field(description="Количество единиц")

order = "Добавьте в заказ 3 стула."

In [ ]:
product_v1 = v1.with_structured_output(Product, method="json_schema").invoke(order)
print(product_v1.model_dump_json(indent=2))
print("Количество как Python int:", product_v1.count)

In [ ]:
product_v2 = v2.with_structured_output(Product, method="json_schema", strict=True).invoke(order)
print(product_v2.model_dump_json(indent=2))
print("Количество как Python int:", product_v2.count)

**Пример объекта — иллюстрация**

```json
{
  "name": "стул",
  "count": 3
}
```

### А если нужен исходный ответ и разобранный объект?

`include_raw=True` возвращает `raw`, `parsed` и `parsing_error`. Это удобно при отладке схемы.

In [ ]:
structured = v2.with_structured_output(Product, method="json_schema", include_raw=True).invoke(order)
answer(structured["raw"], "Исходный JSON модели")
print("Разобранный объект:", structured["parsed"])
print("Ошибка парсинга:", structured["parsing_error"])

### Тот же объект через function calling

Этот метод использует клиентскую функцию для передачи структуры. Сравним его с native JSON Schema.

In [ ]:
function_product_v1 = v1.with_structured_output(Product, method="function_calling").invoke(order)
print(function_product_v1.model_dump_json(indent=2))

In [ ]:
function_product_v2 = v2.with_structured_output(Product, method="function_calling").invoke(order)
print(function_product_v2.model_dump_json(indent=2))

### Низкоуровневый response_format

`bind(response_format=...)` управляет форматом запроса, но возвращает обычный `AIMessage`.
Если нужен готовый Pydantic-объект, используйте `with_structured_output()` выше.

In [ ]:
raw_json = v2.bind(response_format={
    "type": "json_schema", "schema": Product.model_json_schema(), "strict": True,
}).invoke(order)
answer(raw_json)
print("Тип результата:", type(raw_json).__name__)
print("Ручной разбор:", Product.model_validate_json(raw_json.text))

## 5. Инструмент, который действительно исполняется в Python

Модель попросит умножить 17 на 23. Python выполнит вычисление и вернёт **391**.
Затем модель получит `ToolMessage` и сформулирует ответ. Это три отдельных шага, которые можно посмотреть.

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
    """Умножить два целых числа."""
    return a * b

### Шаг 1. Модель выбирает функцию и аргументы — v2

In [ ]:
tool_question = HumanMessage("Вычисли 17 × 23 с помощью multiply и поясни результат одной фразой.")
tool_request = v2.bind_tools([multiply], tool_choice="multiply").invoke([tool_question])
print(json.dumps(tool_request.tool_calls, ensure_ascii=False, indent=2))
print("Сервер вернул tools_state_id:", bool(tool_request.additional_kwargs.get("tools_state_id")))

**Пример запроса инструмента — иллюстрация**

```python
[{"name": "multiply", "args": {"a": 17, "b": 23}, "id": "<ID от сервера>", "type": "tool_call"}]
```
ID условный; в `ToolMessage` нужно перенести ID из настоящего ответа.

### Шаг 2. Выполняем именно запрошенную функцию

Call ID переносим без изменений. Не выбрасывайте исходный `tool_request`: в нём находится состояние провайдера.

Результат передаём как JSON-объект: `json.dumps({"result": calculated})`.
В этом примере не используем `str(calculated)`: адаптер разбирает строку `"391"` как число,
и endpoint из пользовательского запуска ответил 422 на результат инструмента.
Объект соответствует форме результата в интеграционных примерах; исправление требует повторного live-запуска.

In [ ]:
call = tool_request.tool_calls[0]
if call["name"] != "multiply":
    raise ValueError("Неожиданный инструмент")
calculated = multiply.invoke(call["args"])
print("Результат Python:", calculated)
tool_result = ToolMessage(content=json.dumps({"result": calculated}), tool_call_id=call["id"], name=call["name"])

**Результат локального вычисления для a=17, b=23 — иллюстрация**

`391` — это результат Python-функции, не генерация модели.

### Шаг 3. Возвращаем результат модели

In [ ]:
tool_final = v2.bind_tools([multiply], tool_choice="none").invoke(
    [tool_question, tool_request, tool_result])
answer(tool_final)

**Пример финального ответа — иллюстрация**

17 × 23 = 391.

### Для сравнения: полный цикл на v1

Публичный код почти тот же. История создаётся заново: `tools_state_id` v2 нельзя передавать в v1,
а legacy `functions_state_id` — в v2.

In [ ]:
legacy_question = HumanMessage("Вычисли 17 × 23 с помощью multiply и поясни результат одной фразой.")
legacy_request = v1.bind_tools([multiply], tool_choice="multiply").invoke([legacy_question])
print("Запрос инструмента:", legacy_request.tool_calls)
legacy_call = legacy_request.tool_calls[0]
if legacy_call["name"] != "multiply":
    raise ValueError("Неожиданный инструмент")
legacy_value = multiply.invoke(legacy_call["args"])
print("Результат Python:", legacy_value)
legacy_result = ToolMessage(content=json.dumps({"result": legacy_value}), tool_call_id=legacy_call["id"], name=legacy_call["name"])
legacy_final = v1.bind_tools([multiply], tool_choice="none").invoke(
    [legacy_question, legacy_request, legacy_result])
answer(legacy_final, "API v1 · финальный ответ")

### Tool call в stream — v2

Сначала собираем чанки: отдельный чанк может содержать только часть аргументов.
После сборки цикл продолжается обычным `ToolMessage`.

In [ ]:
stream_question = HumanMessage("Вычисли 17 × 23 через multiply.")
stream_request = collect(v2.bind_tools([multiply], tool_choice="multiply").stream([stream_question]))
print("Собранный tool call:", stream_request.tool_calls)
stream_call = stream_request.tool_calls[0]
if stream_call["name"] != "multiply":
    raise ValueError("Неожиданный инструмент")
stream_value = multiply.invoke(stream_call["args"])
print("Результат Python:", stream_value)
stream_result = ToolMessage(content=json.dumps({"result": stream_value}), tool_call_id=stream_call["id"], name=stream_call["name"])
answer(v2.bind_tools([multiply], tool_choice="none").invoke(
    [stream_question, stream_request, stream_result]))

## 6. Reasoning-модель

Reasoning поддерживался и в v1. Здесь сравниваем ответы на одинаковую задачу и смотрим,
появились ли reasoning-блоки в новом контракте. Модель настраивается через `GIGACHAT_V2_REASONING_MODEL`.

In [ ]:
REASONING_MODEL = os.getenv("GIGACHAT_V2_REASONING_MODEL", "GigaChat-2-Reasoning")
reasoning_question = "У Ани 12 монет, у Бори вдвое больше. Вместе они отдали 9 монет. Сколько осталось?"
think_v1 = model(v2=False, model=REASONING_MODEL, reasoning_effort="high", max_tokens=2048)
think_v2 = model(v2=True, model=REASONING_MODEL, reasoning_effort="high", max_tokens=2048)

In [ ]:
reasoning_v1 = think_v1.invoke(reasoning_question)
answer(reasoning_v1, "v1 · ответ на задачу")
print("Reasoning в additional_kwargs:", bool(reasoning_v1.additional_kwargs.get("reasoning_content")))
print("Типы блоков:", [b.get("type") for b in reasoning_v1.content_blocks])

In [ ]:
reasoning_v2 = think_v2.invoke(reasoning_question)
answer(reasoning_v2, "v2 · ответ на задачу")
print("Reasoning в additional_kwargs:", bool(reasoning_v2.additional_kwargs.get("reasoning_content")))
print("Типы блоков:", [b.get("type") for b in reasoning_v2.content_blocks])

**Пример ответа на задачу — иллюстрация**

У Бори 24 монеты. Вместе было 12 + 24 = 36 монет, после передачи 9 осталось 27.

### Reasoning со streaming

In [ ]:
reasoning_stream_v1 = collect(think_v1.stream(reasoning_question))
print("Reasoning получен:", bool(reasoning_stream_v1.additional_kwargs.get("reasoning_content"))
      or any(b.get("type") == "reasoning" for b in reasoning_stream_v1.content_blocks))
details(reasoning_stream_v1)

In [ ]:
reasoning_stream_v2 = collect(think_v2.stream(reasoning_question))
print("Reasoning получен:", bool(reasoning_stream_v2.additional_kwargs.get("reasoning_content"))
      or any(b.get("type") == "reasoning" for b in reasoning_stream_v2.content_blocks))
details(reasoning_stream_v2)

## 7. Настройки генерации в новом контракте

Обычные `temperature`, `max_tokens`, `reasoning_effort` адаптер сам переводит в `model_options`.
При необходимости можно передать native options: они имеют приоритет над обычными аргументами.
Здесь также включаем фильтр и отключаем ranker. Это демонстрация передачи настроек, а не измерение их влияния.

In [ ]:
configured = v2.invoke(
    "Предложи три коротких названия для книжного клуба.",
    model_options={"temperature": 0.7, "top_p": 0.9, "max_tokens": 128,
                   "repetition_penalty": 1.0, "update_interval": 0.5},
    ranker_options={"enabled": False}, profanity_check=True,
)
answer(configured)

**Пример ответа — иллюстрация**

1. Между строк
2. Читаем вместе
3. Книжная полка

## 8. Поиск в интернете силами сервера

`web_search` исполняется на стороне GigaChat. В v1 такого binding нет.
Кроме текста посмотрим источники и события инструмента, если сервер их вернул.

In [ ]:
search = v2.bind_tools([{"type": "web_search"}], tool_choice="web_search")
search_answer = search.invoke("Найди официальный сайт Python. Кратко объясни, что это за язык, и приведи источник.")
answer(search_answer)
for block in search_answer.content_blocks:
    for source in block.get("annotations", []):
        print("Источник:", source)
    if block.get("type") in ("server_tool_call", "server_tool_result"):
        print("Инструмент:", block.get("name"), "событие:", block["type"], "статус:", block.get("status"))

**Пример формы ответа с источником — иллюстрация**

Python — язык программирования общего назначения. Официальный сайт: https://www.python.org/.

В реальном выводе отдельно отображаются возвращённые сервером annotations. Этот пример не подтверждает запуск web_search.

### Поиск со streaming

Серверные tool blocks сохраняются в агрегированном сообщении вместе с текстом.

In [ ]:
search_stream = collect(v2.bind_tools([{"type": "web_search"}], tool_choice="web_search").stream(
    "Найди официальный сайт Python и приведи ссылку."))
print("Блоки результата:", [b.get("type") for b in search_stream.content_blocks])
details(search_stream)

## 9. Code interpreter на сервере

Просим вычислить сумму квадратов от 1 до 20. Локально знаем результат — 2870.
Это другой механизм, чем наш Python-инструмент `multiply`: вычисление запрашивается у провайдера.

In [ ]:
interpreter = v2.bind_tools([{"type": "code_interpreter"}], tool_choice="code_interpreter")
code_answer = interpreter.invoke("Используй code_interpreter: вычисли сумму квадратов целых чисел от 1 до 20.")
answer(code_answer)
print("Для сверки, локальный Python:", sum(i*i for i in range(1, 21)))
print("Блоки результата:", [b.get("type") for b in code_answer.content_blocks])

**Ожидаемый математический результат — иллюстрация**

Сумма квадратов чисел от 1 до 20 равна **2870**. Наличие этого числа в тексте само по себе не доказывает запуск code_interpreter — смотрите server-tool blocks.

### Code interpreter со streaming

In [ ]:
code_stream = collect(v2.bind_tools([{"type": "code_interpreter"}], tool_choice="code_interpreter").stream(
    "Используй code_interpreter: вычисли сумму квадратов целых чисел от 1 до 20."))
print("Блоки результата:", [b.get("type") for b in code_stream.content_blocks])

## 10. Генерируем картинку через image_generate

Встроенный инструмент GigaChat создаёт изображение на сервере. Это отдельный необязательный пример:
для загрузки в следующем разделе уже лежит готовый маленький JPG.
Доступность `image_generate` зависит от endpoint и модели. Сначала получаем ответ с файлом, затем скачиваем его для отображения.

In [ ]:
image_reply = v2.bind_tools(
    [{"type": "image_generate"}], tool_choice="image_generate",
).invoke("Нарисуй жёлтую кружку и два красных яблока на голубом столе. Без надписей.")
answer(image_reply, "Ответ image_generate")
print("Блоки ответа:", [block.get("type") for block in image_reply.content_blocks])
generated_image_ids = [
    block["file_id"] for block in image_reply.content_blocks
    if block.get("type") in ("image", "file") and block.get("file_id")
]
print("ID созданных файлов:", generated_image_ids)

### Показываем созданное изображение

`get_file_content()` возвращает base64. Декодируем его и показываем изображение прямо в notebook.
Если сервер не вернул file ID, смотрите `image_reply.content_blocks` — текстовый ответ сам по себе не подтверждает генерацию.

In [ ]:
import base64
from IPython.display import Image as NotebookImage

if not generated_image_ids:
    raise RuntimeError("image_generate не вернул файл изображения; посмотрите content_blocks выше")
generated_file = v2.get_file_content(generated_image_ids[0])
display(NotebookImage(data=base64.b64decode(generated_file.content), width=384))

## 11. Загружаем маленький JPG и спрашиваем, что на нём

Файл `examples/assets/mug-apples.jpg` уже лежит рядом с notebook: **384 × 384 пикселя**,
жёлтая кружка и два красных яблока. Это синтетическая картинка для примера.
Сначала посмотрим её локально, затем загрузим через Files API и передадим полученный ID модели.
Заново генерировать изображение для этого примера не нужно.

In [ ]:
from IPython.display import Image as NotebookImage

jpg_path = ROOT / "examples" / "assets" / "mug-apples.jpg"
print(f"Файл: {jpg_path.name}; размер: {jpg_path.stat().st_size / 1024:.1f} КБ")
display(NotebookImage(filename=str(jpg_path), width=384))

### Загружаем файл один раз

Эта ячейка отправляет готовый JPG на настроенный GigaChat endpoint. `uploaded.id_` — ID файла,
который можно повторно использовать в запросах. Повторный запуск upload создаёт новую загрузку.

In [ ]:
with jpg_path.open("rb") as image_file:
    uploaded = v2.upload_file((jpg_path.name, image_file.read()), purpose="general")
image_file_id = uploaded.id_
print("Загруженный файл:", image_file_id)

### Описание изображения — v2

In [ ]:
image_question = HumanMessage(content=[
    {"type": "text", "text": "Что на изображении? Назови предметы, их цвета и количество яблок."},
    {"type": "image", "file_id": image_file_id, "mime_type": "image/jpeg"},
])
image_answer_v2 = v2.invoke([image_question])
answer(image_answer_v2, "Описание картинки · v2")

### Та же картинка — v1

Используем уже загруженный файл, второй upload не нужен. Для ручной сверки: кружка жёлтая, яблок два.

In [ ]:
image_answer_v1 = v1.invoke([image_question])
answer(image_answer_v1, "Описание картинки · v1")

## 12. Модель один раз, дальше — только ID диалога

В этом SDK идентификатор называется **`thread_id`**. Получаем его из первого ответа;
`message_id` относится к отдельному сообщению и не используется для продолжения.
Никакой заранее созданный assistant или ID из `.env` здесь не нужен.

Первый запрос передаёт `model=MODEL` и `storage=True`, чтобы сервер сохранил диалог.
Следующие запросы передают только новое сообщение и `storage={"thread_id": conversation_id}`.
Адаптер не отправляет модель из конструктора при продолжении по thread ID.

In [ ]:
first_turn = v2.invoke(
    "Планируем пикник на троих. Один человек не ест молочные продукты. Запомни эти условия.",
    model=MODEL,
    storage=True,
)
answer(first_turn, "Первый ответ")
conversation_id = first_turn.response_metadata.get("thread_id")
if not conversation_id:
    raise RuntimeError("Сервер не вернул thread_id; продолжать без ID нельзя")
print("ID диалога:", conversation_id)

### Второй запрос: модель и историю больше не передаём

In [ ]:
second_turn = v2.invoke(
    "Предложи меню для нашего пикника с учётом ограничения. На сколько человек оно рассчитано?",
    storage={"thread_id": conversation_id},
)
answer(second_turn, "Продолжение по ID")

### Ещё один запрос с тем же ID — теперь streaming

В новом сообщении нет ни количества людей, ни ограничения. Ответ должен учитывать их из сохранённого диалога.

In [ ]:
third_turn = collect(v2.stream(
    "Составь короткий список покупок для этого меню.",
    storage={"thread_id": conversation_id},
))

## 13. Небольшая цепочка LangChain

Шаблон запроса → модель → строковый parser. Удобно, когда нужен готовый текст, а не `AIMessage`.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты редактор. Перепиши текст вежливо, коротко и без канцелярита."),
    ("human", "{text}"),
])
source_text = "Вам необходимо осуществить предоставление отчётности до наступления пятницы."

In [ ]:
chain_v1 = prompt | v1 | StrOutputParser()
edited_v1 = chain_v1.invoke({"text": source_text})
print("Было:", source_text)
print("v1:", edited_v1)

In [ ]:
chain_v2 = prompt | v2 | StrOutputParser()
edited_v2 = chain_v2.invoke({"text": source_text})
print("Было:", source_text)
print("v2:", edited_v2)

## Что стоит запомнить про v1 и v2

| Сценарий | v1 | v2 |
|---|---|---|
| Обычные запросы, async и streaming | Работают через legacy SDK | Тот же интерфейс LangChain, новые ресурсы SDK |
| Structured output | function_calling и json_schema | function_calling и native json_schema |
| Клиентские tools | functions / functions_state_id | tools / tools_state_id; call ID переносится как есть |
| Reasoning | Уже поддерживался | Новый mapping и stream conversion |
| web_search / code_interpreter / image_generate | Такого binding нет | Встроенные server tools, если доступны на endpoint |
| Продолжение по ID | — | Модель в первом запросе; затем storage с полученным thread_id |
| Загрузка JPG | upload_file → file ID → сообщение | Тот же file ID, primary image block |

V2 поддерживает несколько клиентских tool calls с отдельными ID, `parallel_tool_calls=True`
и `tool_choice="any"` / `"required"`. На каждый вызов верните отдельный `ToolMessage` с его ID.
Tool-истории между маршрутами не смешиваются. Параллельные server tools без различимых ID не поддерживаются.
Полные циклы показаны в [примерах инструментов](sdk_contract_alignment/README.md).

**Сравнение здесь функциональное.** Мы смотрим ответы и структуру результатов на одинаковых примерах.
Это не бенчмарк скорости или качества: для него нужны повторы и отдельная методика.

### Откуда взяты примеры

- [LangChain: модели, streaming, tools и structured output](https://docs.langchain.com/oss/python/langchain/models) — рецепты сверены через Context7.
- [LangChain: structured output](https://docs.langchain.com/oss/python/langchain/structured-output).
- [CHANGELOG 0.5.2a1](../libs/gigachat/CHANGELOG.md) и [README API v2](../libs/gigachat/README.md#api-v2-v2chatcompletions).
- [Контрактные тесты адаптера](../libs/gigachat/tests/unit_tests/chat_models/primary) — для подробных assertions и ошибок протокола.

Перед публикацией очистите ответы, содержащие содержимое ваших файлов или приватных диалогов.